# 论文 14：联合学习对齐和翻译的神经机器翻译
## Dzmitry Bahdanau, Kyunghyun Cho, Yoshua Bengio（2014）

### 原始注意力机制

本文提出了**注意力机制**，这是深度学习中最重要的创新之一，比 Transformer 早三年出现。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 问题：固定长度的上下文向量

传统 Seq2Seq 将整个输入压缩成单个向量，从而形成信息瓶颈。

In [ ]:
def softmax(x, axis=-1):
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

class EncoderRNN:
    '双向 RNN 编码器。'
    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size
        
        # 前向 RNN
        self.W_fwd = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        self.b_fwd = np.zeros((hidden_size, 1))
        
        # 反向 RNN
        self.W_bwd = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        self.b_bwd = np.zeros((hidden_size, 1))
    
    def forward(self, inputs):
        """inputs：形状为 (input_size, 1) 的向量列表
        返回：双向隐藏状态列表，每个状态形状为 (2*hidden_size, 1)"""
        seq_len = len(inputs)
        
        # 前向传播
        h_fwd = []
        h = np.zeros((self.hidden_size, 1))
        for x in inputs:
            concat = np.vstack([x, h])
            h = np.tanh(np.dot(self.W_fwd, concat) + self.b_fwd)
            h_fwd.append(h)
        
        # 反向遍历输入序列
        h_bwd = []
        h = np.zeros((self.hidden_size, 1))
        for x in reversed(inputs):
            concat = np.vstack([x, h])
            h = np.tanh(np.dot(self.W_bwd, concat) + self.b_bwd)
            h_bwd.append(h)
        h_bwd = list(reversed(h_bwd))
        
        # 拼接前向和反向隐藏状态
        annotations = [np.vstack([h_f, h_b]) for h_f, h_b in zip(h_fwd, h_bwd)]
        
        return annotations

print("Bidirectional Encoder created")

## Bahdanau 注意力机制

关键创新：联合学习对齐与翻译。

**注意力分数**：$e_{ij} = a(s_{i-1}, h_j)$，其中 $s$ 是解码器状态，$h$ 是编码器表示

**注意力权重**：$\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_k \exp(e_{ik})}$

**上下文向量**：$c_i = \sum_j \alpha_{ij} h_j$

In [ ]:
class BahdanauAttention:
    '加性注意力机制。'
    def __init__(self, hidden_size, annotation_size):
        self.hidden_size = hidden_size
        
        # 注意力参数
        self.W_a = np.random.randn(hidden_size, hidden_size) * 0.01
        self.U_a = np.random.randn(hidden_size, annotation_size) * 0.01
        self.v_a = np.random.randn(1, hidden_size) * 0.01
    
    def forward(self, decoder_hidden, encoder_annotations):
        """decoder_hidden：(hidden_size, 1)，当前解码器状态 s_{i-1}
        encoder_annotations：形状为 (annotation_size, 1) 的列表，包含所有编码器状态 h_j
        
        返回：
        context：(annotation_size, 1)，编码器表示的加权和
        attention_weights：(seq_len,)，注意力分布"""
        scores = []
        
        # 计算每个位置的注意力分数
        for h_j in encoder_annotations:
            # e_ij = v_a^T * tanh(W_a * s_{i-1} + U_a * h_j)
            score = np.dot(self.v_a, np.tanh(
                np.dot(self.W_a, decoder_hidden) + 
                np.dot(self.U_a, h_j)
            ))
            scores.append(score[0, 0])
        
        # Softmax 获取注意力权重
        scores = np.array(scores)
        attention_weights = softmax(scores)
        
        # 将上下文向量计算为加权和
        context = sum(alpha * h for alpha, h in zip(attention_weights, encoder_annotations))
        
        return context, attention_weights

print("Bahdanau Attention mechanism created")

## 具有注意力的解码器

In [ ]:
class AttentionDecoder:
    '带 Bahdanau 注意力的 RNN 解码器。'
    def __init__(self, output_size, hidden_size, annotation_size):
        self.hidden_size = hidden_size
        self.output_size = output_size
        
        # 注意力机制
        self.attention = BahdanauAttention(hidden_size, annotation_size)
        
        # RNN 输入由前一个输出和上下文向量组成
        input_size = output_size + annotation_size
        self.W_dec = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        self.b_dec = np.zeros((hidden_size, 1))
        
        # 输出层
        self.W_out = np.random.randn(output_size, hidden_size + annotation_size + output_size) * 0.01
        self.b_out = np.zeros((output_size, 1))
    
    def step(self, prev_output, decoder_hidden, encoder_annotations):
        """执行单个解码步骤。
        
        prev_output：(output_size, 1)，前一个输出词元
        decoder_hidden：(hidden_size, 1)，前一个解码器状态
        encoder_annotations：形状为 (annotation_size, 1) 的编码器状态列表
        
        返回：
        output：(output_size, 1)，预测输出分布
        new_hidden：(hidden_size, 1)，新的解码器状态
        attention_weights：注意力分布"""
        # 计算注意力和上下文
        context, attention_weights = self.attention.forward(decoder_hidden, encoder_annotations)
        
        # 解码器 RNN: s_i = f(s_{i-1}, y_{i-1}, c_i)
        rnn_input = np.vstack([prev_output, context])
        concat = np.vstack([rnn_input, decoder_hidden])
        new_hidden = np.tanh(np.dot(self.W_dec, concat) + self.b_dec)
        
        # 输出：y_i = g(s_i, y_{i-1}, c_i)
        output_input = np.vstack([new_hidden, context, prev_output])
        output = np.dot(self.W_out, output_input) + self.b_out
        
        return output, new_hidden, attention_weights
    
    def forward(self, encoder_annotations, max_length=20, start_token=None):
        '执行完整解码过程。'
        if start_token is None:
            start_token = np.zeros((self.output_size, 1))
        
        outputs = []
        attention_history = []
        
        # 初始化
        decoder_hidden = np.zeros((self.hidden_size, 1))
        prev_output = start_token
        
        for _ in range(max_length):
            output, decoder_hidden, attention_weights = self.step(
                prev_output, decoder_hidden, encoder_annotations
            )
            
            outputs.append(output)
            attention_history.append(attention_weights)
            
            # 下一个输入是当前输出（贪婪解码）
            prev_output = output
        
        return outputs, attention_history

print("Attention Decoder created")

## 使用注意力机制完成 Seq2Seq

In [ ]:
class Seq2SeqWithAttention:
    def __init__(self, input_vocab_size, output_vocab_size, hidden_size=32):
        self.input_vocab_size = input_vocab_size
        self.output_vocab_size = output_vocab_size
        self.hidden_size = hidden_size
        
        # 嵌入层
        self.input_embedding = np.random.randn(input_vocab_size, hidden_size) * 0.01
        self.output_embedding = np.random.randn(output_vocab_size, hidden_size) * 0.01
        
        # 编码器是双向的，因此编码器表示维度为 2*hidden_size
        self.encoder = EncoderRNN(hidden_size, hidden_size)
        
        # 带有注意力的解码器
        annotation_size = 2 * hidden_size
        self.decoder = AttentionDecoder(hidden_size, hidden_size, annotation_size)
    
    def translate(self, input_sequence, max_output_length=15):
        """将输入序列转换为输出序列
        
        input_sequence：词元索引列表"""
        # 嵌入输入
        embedded = [self.input_embedding[idx:idx+1].T for idx in input_sequence]
        
        # 编码
        annotations = self.encoder.forward(embedded)
        
        # 解码
        start_token = self.output_embedding[0:1].T  # 使用第一个词元作为起始标记
        outputs, attention_history = self.decoder.forward(
            annotations, max_length=max_output_length, start_token=start_token
        )
        
        return outputs, attention_history, annotations

# 创建模型
input_vocab_size = 20   # 源语言词汇
output_vocab_size = 20  # 目标语言词汇
model = Seq2SeqWithAttention(input_vocab_size, output_vocab_size, hidden_size=16)

print(f"Seq2Seq with Attention created")
print(f"Input vocab: {input_vocab_size}")
print(f"Output vocab: {output_vocab_size}")

## 合成翻译任务测试

In [ ]:
# 简单的合成任务：逆序
# 输入：[1,2,3,4,5]
# 输出：[5,4,3,2,1]

input_seq = [1, 2, 3, 4, 5, 6, 7]
outputs, attention_history, annotations = model.translate(input_seq, max_output_length=len(input_seq))

print(f"Input sequence: {input_seq}")
print(f"Number of output steps: {len(outputs)}")
print(f"Number of attention distributions: {len(attention_history)}")
print(f"Encoder annotations shape: {len(annotations)} x {annotations[0].shape}")

## 可视化注意力权重

核心认识：可以直接观察模型在关注哪些输入位置。

In [ ]:
# 将注意力历史转换为矩阵
attention_matrix = np.array(attention_history)  # （output_len、input_len）

plt.figure(figsize=(10, 8))
plt.imshow(attention_matrix, cmap='Blues', aspect='auto', interpolation='nearest')
plt.colorbar(label='Attention Weight')
plt.xlabel('Input Position (Source)')
plt.ylabel('Output Position (Target)')
plt.title('Bahdanau Attention Alignment Matrix')

# 添加网格
plt.xticks(range(len(input_seq)), [f'x{i+1}' for i in input_seq])
plt.yticks(range(len(outputs)), [f'y{i+1}' for i in range(len(outputs))])

plt.tight_layout()
plt.show()

print("\nAttention patterns show which input positions influence each output.")
print("Brighter cells = higher attention weight.")

## 观察每个解码步骤的注意力

In [ ]:
# 可视化特定解码器步骤的注意力分布
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

steps_to_show = min(8, len(attention_history))

for i in range(steps_to_show):
    axes[i].bar(range(len(input_seq)), attention_history[i])
    axes[i].set_title(f'Output Step {i+1}')
    axes[i].set_xlabel('Input Position')
    axes[i].set_ylabel('Attention Weight')
    axes[i].set_ylim(0, 1)
    axes[i].set_xticks(range(len(input_seq)))
    axes[i].set_xticklabels([f'x{j+1}' for j in input_seq], fontsize=8)
    axes[i].grid(True, alpha=0.3, axis='y')

plt.suptitle('Attention Distribution at Each Decoding Step', fontsize=14)
plt.tight_layout()
plt.show()

print("Each decoder step focuses on different input positions!")

## 比较：有注意力和无注意力

In [ ]:
# 模拟固定上下文 seq2seq（无注意）
def fixed_context_attention(seq_len):
    '模拟仅关注最后的编码器状态'
    weights = np.zeros(seq_len)
    weights[-1] = 1.0  # 只关注最后一个位置
    return weights

# 创建比较
input_length = len(input_seq)
output_length = len(outputs)

# 固定上下文
fixed_attention = np.array([fixed_context_attention(input_length) for _ in range(output_length)])

# 绘制对比结果
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 没有注意力（固定上下文）
im1 = ax1.imshow(fixed_attention, cmap='Blues', aspect='auto', vmin=0, vmax=1)
ax1.set_xlabel('Input Position')
ax1.set_ylabel('Output Position')
ax1.set_title('Without Attention (Fixed Context)\nAll decoder steps see only last encoder state')
plt.colorbar(im1, ax=ax1)

# Bahdanau 注意力
im2 = ax2.imshow(attention_matrix, cmap='Blues', aspect='auto', vmin=0, vmax=1)
ax2.set_xlabel('Input Position')
ax2.set_ylabel('Output Position')
ax2.set_title('With Bahdanau Attention\nEach decoder step attends to different positions')
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

print("\nKey Difference:")
print("  Without attention: Information bottleneck at last encoder state")
print("  With attention: Dynamic access to all encoder states")

## 注意力机制变体

In [ ]:
def bahdanau_score(s, h, W_a, U_a, v_a):
    '加性/拼接注意力（Bahdanau）。'
    return np.dot(v_a.T, np.tanh(np.dot(W_a, s) + np.dot(U_a, h)))[0, 0]

def dot_product_score(s, h):
    '点积注意力 (Luong)'
    return np.dot(s.T, h)[0, 0]

def scaled_dot_product_score(s, h):
    '缩放点积（Transformer 型）'
    d_k = s.shape[0]
    return np.dot(s.T, h)[0, 0] / np.sqrt(d_k)

# 比较评分函数
s = np.random.randn(16, 1)
h = np.random.randn(32, 1)
W_a = np.random.randn(16, 16)
U_a = np.random.randn(16, 32)
v_a = np.random.randn(1, 16)

print("Attention Score Functions:")
print(f"  Bahdanau (additive): score = v^T tanh(W*s + U*h)")
print(f"  Dot product: score = s^T h")
print(f"  Scaled dot product: score = s^T h / sqrt(d_k)")
print(f"\nBahdanau is more expressive but has more parameters.")

## 要点

### 注意力机制解决的问题
- **固定长度上下文**：整个输入压缩为单个向量
- **信息瓶颈**：长序列丢失信息
- **没有对齐**：解码器不知道要关注哪个输入

### Bahdanau 注意力的创新
1. **动态上下文**：每个解码器步骤都不同
2. **软对齐**：学习对齐源和目标
3. **所有编码器状态**：解码器可以访问所有状态，而不仅仅是最后一个状态

### 工作原理
```
1. Encoder produces annotations h_1, ..., h_T
2. For each decoder step i:
   a. Compute attention scores: e_ij = score(s_{i-1}, h_j)
   b. Normalize to weights: α_ij = softmax(e_ij)
   c. Compute context: c_i = Σ α_ij * h_j
   d. Generate output: y_i = f(s_i, c_i, y_{i-1})
```

### Bahdanau 与 Luong 注意力对比
| 特征 | Bahdanau（2014） | Luong（2015） |
|---------|----------------|---------------|
| 评分函数 | 加法：v·tanh(W·s + U·h) | 乘法：s·h |
| 使用的解码器状态 | s_{i-1}（前一状态） | s_i（当前状态） |
| 全局/局部 | 仅全局注意力 | 两者均可 |

### 数学公式：

**注意力分数（对齐模型）**：
$$e_{ij} = v_a^T \tanh(W_a s_{i-1} + U_a h_j)$$

**注意力权重**：
$$\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k=1}^{T_x} \exp(e_{ik})}$$

**上下文向量**：
$$c_i = \sum_{j=1}^{T_x} \alpha_{ij} h_j$$

**解码器**：
$$s_i = f(s_{i-1}, y_{i-1}, c_i)$$
$$p(y_i | y_{<i}, x) = g(s_i, y_{i-1}, c_i)$$

### 影响：
- **革命性的 NMT**：BLEU 分数大幅跃升
- **可解释性**：可以可视化对齐
- **Transformer 的基础**：为纯注意力架构奠定基础
- **超越 NMT**：扩展到视觉、语音等领域

### 为什么它有效：
1. **解决瓶颈**：可变长度上下文
2. **学习对齐**：不需要单独的对齐模型
3. **可微分**：端到端训练
4. **适用于长序列**：注意力不会衰减

### 现代视角：
- Transformer 使用**自注意力**，即在同一个序列内部计算注意力
- 缩放点积现已成为标准（更简单、更快）
- 多头注意力捕捉不同的关系
- Bahdanau 的核心思想仍然成立：**关注与当前任务相关的信息**